Gauntlet #2: The Ellipsis (...) and High‑Dimensional Indexing

## 📘 Day 2, Topic 5 (Gauntlet #2): Ellipsis `...` and High-Dimensional Indexing

### The Problem With High-Dimensional Arrays
When you have a 5D array of shape `(2, 3, 4, 5, 6)`, writing explicit indices for every dimension is verbose and brittle:
```python
arr[0, :, :, :, :]   # select first block, keep all other dims
```
The **Ellipsis (`...`)** is NumPy's shorthand for "fill in as many `:` slices as needed".

---

### What `...` Does
`...` expands to however many `:` colons are needed to fill the remaining dimensions.

```python
arr_5d.shape = (2, 3, 4, 5, 6)

arr_5d[0, ...]       # = arr_5d[0, :, :, :, :]   → shape (3, 4, 5, 6)
arr_5d[..., 2]       # = arr_5d[:, :, :, :, 2]   → shape (2, 3, 4, 5)
arr_5d[0, ..., 3, :] # = arr_5d[0, :, :, 3, :]   → shape (3, 4, 6)
```

**Rules:**
- Only **one `...`** is allowed per index expression
- It can appear at the start, middle, or end
- It expands to zero or more `:` as needed

---

### Predicting Output Shape — Step by Step

To predict the shape after indexing:
1. Write out the full index with `...` expanded to explicit `:`
2. Remove dimensions where you used a **single integer** (they collapse)
3. Keep dimensions where you used `:` or a **slice** (they survive)
4. A `np.newaxis` **adds** a new dimension of size 1

```
arr_5d shape: (2, 3, 4, 5, 6)
Index: arr_5d[0, ..., 3, :]
Expand: arr_5d[0, :, :, 3, :]
         ^              ^       ← integers → collapse these dims
            ^  ^     ^          ← slices → keep these dims
Result shape: (3, 4, 6)
```

---

### Quick Shape Prediction Reference

| Index expression | Expanded form | Resulting shape |
|-----------------|---------------|-----------------|
| `arr[0, ...]` | `arr[0, :, :, :, :]` | `(3, 4, 5, 6)` |
| `arr[..., 2]` | `arr[:, :, :, :, 2]` | `(2, 3, 4, 5)` |
| `arr[0, ..., 3, :]` | `arr[0, :, :, 3, :]` | `(3, 4, 6)` |
| `arr[..., 1:4, 2]` | `arr[:, :, :, 1:4, 2]` | `(2, 3, 4, 3)` |
| `arr[0, 1, ..., 2:5]` | `arr[0, 1, :, :, 2:5]` | `(4, 5, 3)` |
| `arr[:, :, 2, ...]` | `arr[:, :, 2, :, :]` | `(2, 3, 5, 6)` |

---

### `np.newaxis` — Adding a Dimension
`np.newaxis` (same as `None`) **inserts** a new axis of size 1 at that position. This is used to make arrays broadcast-compatible.

```python
arr_5d[..., np.newaxis, 2, :]
# Step 1: expand ... for remaining dims needed
# Step 2: np.newaxis adds a size-1 dimension
# Result: shape (2, 3, 4, 1, 6)
```

| Operation | Shape change |
|-----------|-------------|
| `arr[np.newaxis, :]` | Prepends a 1 |
| `arr[:, np.newaxis]` | Inserts a 1 after 1st dim |
| `arr[..., np.newaxis]` | Appends a 1 at the end |

---

### Why `...` Matters in Practice
When writing **generic functions** that must work on arrays of any number of dimensions, `...` lets you index the first or last axis without knowing `ndim`:

```python
def get_last_channel(arr):
    return arr[..., 0]   # works on (H, W, C), (B, H, W, C), or any shape

def get_first_sample(arr):
    return arr[0, ...]   # works on (N, ...) arrays of any depth
```
This is especially common in deep learning code where batch dimensions vary.

In [1]:
import numpy as np

In [2]:
arr_5d = np.arange(2 * 3 * 4 * 5 * 6).reshape(2, 3, 4, 5, 6)


In [3]:
result = arr_5d[0, ...] #dim: 4, shape: (3, 4, 5, 6 )
print(result.shape)
print(result.ndim)

(3, 4, 5, 6)
4


In [4]:
result = arr_5d[..., 2] # dim: 4, shape: 2, 3, 4, 5
print(result.shape)
print(result.ndim)

(2, 3, 4, 5)
4


In [5]:
result = arr_5d[0, ..., 3, :] # dim: 3, shape: 3, 4, 6
print(result.shape)
print(result.ndim)

(3, 4, 6)
3


In [6]:
result = arr_5d[..., 1:4, 2] # dim: 4, shape: 2, 3, 4, 3
print(result.shape)
print(result.ndim)

(2, 3, 4, 3)
4


In [7]:
result = arr_5d[0, 1, ..., 2:5] # dim: 3, shape: 4, 5, 3
print(result.shape)
print(result.ndim)

(4, 5, 3)
3


In [8]:
result = arr_5d[:, :, 2, ...] #dim: 4, shape: 2, 3, 5, 6
print(result.shape)
print(result.ndim)

(2, 3, 5, 6)
4


In [9]:
result = arr_5d[..., np.newaxis, 2, :] # dim: 5, shape:could'nt predict it 
print(result.shape)
print(result.ndim)

(2, 3, 4, 1, 6)
5
